In [ ]:
# COLAB SETUP (uncomment this block when running on Google Colab)
# -----------------------------------------------------------------------------
# IN_COLAB = 'google.colab' in sys.modules
# if IN_COLAB:
#     !pip install -q transformers datasets torch torchvision pillow mlflow ipywidgets jiwer
#     # Define inline functions (no local project imports on Colab)
#     def compute_cer(ref: str, hyp: str) -> float:
#         from jiwer import cer
#         return cer(ref, hyp) if ref and hyp else 1.0
#     def compute_wer(ref: str, hyp: str) -> float:
#         from jiwer import wer
#         return wer(ref, hyp) if ref and hyp else 1.0
#     def apply_gaussian_blur(img, sigma):
#         from PIL import ImageFilter
#         return img.filter(ImageFilter.GaussianBlur(radius=sigma))
#     def apply_rotation(img, degrees):
#         return img.rotate(degrees, expand=False, fillcolor=(255,255,255))
#     def apply_jpeg_compression(img, quality):
#         import io
#         buf = io.BytesIO()
#         img.save(buf, format='JPEG', quality=quality)
#         buf.seek(0)
#         return Image.open(buf).convert('RGB')
# -----------------------------------------------------------------------------

# Suppress verbose warnings from transformers (pooler weights, etc.}

# OCR Model Comparison: From Tiny to SOTA

**Goal**: Compare OCR models across the size/accuracy spectrum

## Key Insight: Line-Level vs Document-Level

| Type | Models | Input | Use Case |
|------|--------|-------|----------|
| **Line-Level** | TrOCR | Cropped text line | Need preprocessing/segmentation first |
| **Document-Level** | Donut, Florence-2, GOT-OCR2, Chandra | Full page | End-to-end, no cropping needed |

## Model Registry

### Line-Level OCR (need text region cropping)
| Model | Vendor | Size | Reference |
|-------|--------|------|-----------|
| [TrOCR-printed](https://huggingface.co/microsoft/trocr-base-printed) | Microsoft | 0.33B | [arXiv:2109.10282](https://arxiv.org/abs/2109.10282) |
| [TrOCR-handwritten](https://huggingface.co/microsoft/trocr-base-handwritten) | Microsoft | 0.33B | Same paper |

### Document-Level OCR (full page capable)
| Model | Vendor | Size | Reference |
|-------|--------|------|-----------|
| [Donut](https://huggingface.co/naver-clova-ix/donut-base) | Naver | 0.2B | [arXiv:2111.15664](https://arxiv.org/abs/2111.15664) |
| [Florence-2](https://huggingface.co/microsoft/Florence-2-base) | Microsoft | 0.23B | [arXiv:2311.06242](https://arxiv.org/abs/2311.06242) |
| [GOT-OCR2](https://huggingface.co/stepfun-ai/GOT-OCR2_0) | StepFun | 0.6B | [GitHub](https://github.com/Ucas-HaoranWei/GOT-OCR2.0) |
| [Qwen2-VL](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct) | Alibaba | 2B | [arXiv:2409.12191](https://arxiv.org/abs/2409.12191) |
| [Chandra](https://huggingface.co/datalab-to/chandra) | Datalab | 9B | [Model Card](https://huggingface.co/datalab-to/chandra) |

## Dataset Progression (Clean → Noisy)

| Dataset | Type | Quality |
|---------|------|---------|
| Synthetic | Generated | Perfect (golden set) |
| [md_invoices](https://huggingface.co/datasets/Am0MuK/md_invoices) | Invoices | Clean, structured |
| [XFUND](https://huggingface.co/datasets/nnul/xfund-multilingual) | Forms | Multilingual, annotated |
| [FUNSD](https://huggingface.co/datasets/nielsr/funsd) | Forms | English, annotated |
| [CORD](https://huggingface.co/datasets/naver-clova-ix/cord-v2) | Receipts | Real-world |
| [scanned_receipts](https://huggingface.co/datasets/Voxel51/scanned_receipts) | Receipts | Noisy, real scans |

## What You'll Learn
1. **Model Comparison** - Which model for which use case?
2. **Dataset Impact** - How data quality affects accuracy
3. **Robustness Testing** - What breaks each model
4. **Cascade Pattern** - Fast model → fallback to accurate

In [ ]:
# =============================================================================
# SETUP - Run this first!
# =============================================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Some weights of.*were not initialized")
import transformers

transformers.logging.set_verbosity_error()

# Add project root to path (local development)
project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Core imports
# Interactive widgets (like Databricks!)
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from PIL import Image, ImageDraw, ImageFont
from projects.ocr_pipeline.project.preprocess import (
    apply_gaussian_blur,
    apply_jpeg_compression,
    apply_rotation,
)

# Project imports (skip on Colab - use inline functions above)
from ml_portfolio.metrics.ocr import compute_cer

# Detect hardware
device = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device: {device}")
print(f"Project root: {project_root}")

## 1. Model Selection

Choose your model based on **input type** and **size/accuracy tradeoff**:

### Line-Level OCR (cropped text lines only)
| Model | Size | Speed | Best For |
|-------|------|-------|----------|
| TrOCR-printed | 0.33B | Fast | Clean printed text lines |
| TrOCR-handwritten | 0.33B | Fast | Handwritten text lines |

### Document-Level OCR (full pages)
| Model | Size | Speed | Best For |
|-------|------|-------|----------|
| Donut | 0.2B | Fast | Document understanding (CORD-trained) |
| Florence-2 | 0.23B | Fast | Multi-task VLM |
| GOT-OCR2 | 0.6B | Medium | General OCR - very capable! |
| Qwen2-VL | 2B | Medium | Multi-modal (non-Microsoft) |
| Chandra | 9B | Slow | SOTA accuracy ceiling |

⚠️ **TrOCR on full pages = garbage output** (see results below)

⚠️ **Chandra requires ~20GB+ VRAM** - use for benchmarking only, or run on cloud GPU.

In [ ]:
# =============================================================================
# MODEL REGISTRY - Available models and their configurations
# =============================================================================
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoProcessor,
    DonutProcessor,
    Qwen2VLForConditionalGeneration,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)
from transformers import (
    VisionEncoderDecoderModel as DonutModel,
)

# Model registry with metadata (references in header cell)
# Organized by capability: Line-level → Document-level → Heavy
MODELS = {
    # === LINE-LEVEL OCR (need text region cropping) ===
    "TrOCR-base-printed (0.33B) [Microsoft]": {
        "id": "microsoft/trocr-base-printed",
        "type": "trocr",
        "size": "0.33B",
        "level": "line",
        "note": "Best for clean printed text lines",
    },
    "TrOCR-base-handwritten (0.33B) [Microsoft]": {
        "id": "microsoft/trocr-base-handwritten",
        "type": "trocr",
        "size": "0.33B",
        "level": "line",
        "note": "For handwritten text lines",
    },
    # === DOCUMENT-LEVEL OCR (full page capable) ===
    "Donut-base (0.2B) [Naver]": {
        "id": "naver-clova-ix/donut-base",
        "type": "donut",
        "size": "0.2B",
        "level": "document",
        "note": "End-to-end document understanding",
    },
    "Florence-2-base (0.23B) [Microsoft]": {
        "id": "microsoft/Florence-2-base",
        "type": "florence",
        "size": "0.23B",
        "task_prompt": "<OCR>",
        "level": "document",
        "note": "Multi-task VLM",
    },
    "GOT-OCR2 (0.6B) [StepFun]": {
        "id": "stepfun-ai/GOT-OCR2_0",
        "type": "got_ocr",
        "size": "0.6B",
        "level": "document",
        "note": "General OCR Theory - very capable!",
    },
    # === MULTI-MODAL / HEAVY ===
    "Qwen2-VL-2B (2B) [Alibaba]": {
        "id": "Qwen/Qwen2-VL-2B-Instruct",
        "type": "qwen2vl",
        "size": "2B",
        "level": "document",
        "note": "Multi-modal, non-Microsoft alternative",
    },
    "Chandra (9B) [Datalab]": {
        "id": "datalab-to/chandra",
        "type": "chandra",
        "size": "9B",
        "level": "document",
        "warning": "Requires ~20GB+ VRAM!",
        "note": "SOTA accuracy ceiling",
    },
}

print(f"Available models: {len(MODELS)}")
print("Line-level:", sum(1 for m in MODELS.values() if m.get("level") == "line"))
print("Document-level:", sum(1 for m in MODELS.values() if m.get("level") == "document"))

In [ ]:
# =============================================================================
# MODEL LOADER - Widget and loading function
# =============================================================================

# Global state
model = None
processor = None
current_model_type = None

# Create selector widget
model_selector = widgets.Dropdown(
    options=list(MODELS.keys()),
    value="TrOCR-base-printed (0.33B) [Microsoft]",
    description="Model:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="380px"),
)
load_btn = widgets.Button(description="Load Model", button_style="primary", icon="download")
status_out = widgets.Output()


def on_load_click(b):
    """Load selected model and processor."""
    global model, processor, current_model_type
    with status_out:
        clear_output()
        model_info = MODELS[model_selector.value]
        model_id = model_info["id"]
        model_type = model_info["type"]

        if model_info.get("warning"):
            print(f"⚠️ {model_info['warning']}")

        print(f"Loading {model_id}...")
        print(f"Type: {model_type} | Size: {model_info['size']}")

        try:
            if model_type == "trocr":
                processor = TrOCRProcessor.from_pretrained(model_id)
                model = VisionEncoderDecoderModel.from_pretrained(model_id)
            elif model_type == "florence":
                processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
                model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
            elif model_type == "qwen2vl":
                processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
                model = Qwen2VLForConditionalGeneration.from_pretrained(
                    model_id, trust_remote_code=True
                )
            elif model_type == "donut":
                processor = DonutProcessor.from_pretrained(model_id)
                model = DonutModel.from_pretrained(model_id)
            elif model_type == "chandra":
                # Chandra uses AutoModel, not AutoModelForCausalLM
                model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
                processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
                model.processor = processor  # Attach for chandra library

            elif model_type == "got_ocr":
                # GOT-OCR2 - General OCR Theory
                from transformers import AutoTokenizer

                tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
                model = AutoModel.from_pretrained(
                    model_id, trust_remote_code=True, low_cpu_mem_usage=True, use_safetensors=True
                )
                processor = tokenizer  # GOT uses tokenizer as processor

            model = model.to(device)
            model.eval()
            current_model_type = model_type

            params = sum(p.numel() for p in model.parameters()) / 1e6
            print(f"\n✅ Loaded: {params:.0f}M parameters on {device}")

        except Exception as e:
            print(f"\n❌ Error loading model: {e}")
            print("   Try a smaller model or check GPU memory.")


load_btn.on_click(on_load_click)
display(widgets.HBox([model_selector, load_btn]))
display(status_out)

## 2. Dataset Selection

Choose your evaluation dataset. Progress from **clean → noisy** to understand model limits.

| Dataset | Type | Quality | Reference |
|---------|------|---------|-----------|
| Synthetic | Generated | Perfect | Local golden set |
| [md_invoices](https://huggingface.co/datasets/Am0MuK/md_invoices) | Invoices | Clean | Invoice→Markdown |
| [XFUND](https://huggingface.co/datasets/nnul/xfund-multilingual) | Forms | Annotated | Multilingual forms |
| [FUNSD](https://huggingface.co/datasets/nielsr/funsd) | Forms | Annotated | English forms |
| [CORD](https://huggingface.co/datasets/naver-clova-ix/cord-v2) | Receipts | Real-world | Donut-compatible |
| [scanned_receipts](https://huggingface.co/datasets/Voxel51/scanned_receipts) | Receipts | Noisy | Robustness testing |

In [ ]:
# =============================================================================
# DATASET SELECTOR & LOADER
# =============================================================================
from datasets import load_dataset

# Dataset registry (references in header cell)
DATASETS = {
    "Synthetic (Golden Set)": {"type": "synthetic", "quality": "perfect"},
    "md_invoices (Clean)": {"type": "huggingface", "id": "Am0MuK/md_invoices", "quality": "clean"},
    "XFUND (Multilingual Forms)": {
        "type": "huggingface",
        "id": "nnul/xfund-multilingual",
        "quality": "annotated",
        "image_key": "image",
        "text_key": "words",
    },
    "FUNSD (English Forms)": {
        "type": "huggingface",
        "id": "nielsr/funsd",
        "quality": "annotated",
        "image_key": "image",
        "text_key": "words",
    },
    "CORD (Receipts)": {
        "type": "huggingface",
        "id": "naver-clova-ix/cord-v2",
        "quality": "real-world",
        "image_key": "image",
        "text_key": "ground_truth",
    },
    "scanned_receipts (Noisy)": {
        "type": "huggingface",
        "id": "Voxel51/scanned_receipts",
        "quality": "noisy",
    },
}

print(f"Available datasets: {len(DATASETS)}")

# Dataset selector widget
dataset_selector = widgets.Dropdown(
    options=list(DATASETS.keys()),
    value="Synthetic (Golden Set)",
    description="Dataset:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="350px"),
)

sample_limit = widgets.IntSlider(
    value=5,
    min=1,
    max=50,
    step=1,
    description="Samples:",
    style={"description_width": "60px"},
)

load_data_btn = widgets.Button(description="Load Dataset", button_style="success", icon="database")
data_status = widgets.Output()

# Global data storage
current_samples = []


def create_text_image(
    text: str, width: int = 400, height: int = 50, font_size: int = 32
) -> Image.Image:
    """Create a synthetic text image with known ground truth."""
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", font_size)
    except:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), text, font=font)
    x = (width - (bbox[2] - bbox[0])) // 2
    y = (height - (bbox[3] - bbox[1])) // 2
    draw.text((x, y), text, fill="black", font=font)
    return img


def load_dataset_samples(b):
    global current_samples
    with data_status:
        clear_output()
        dataset_info = DATASETS[dataset_selector.value]
        n_samples = sample_limit.value

        print(f"Loading {dataset_selector.value}...")
        print(f"Quality: {dataset_info['quality']}")

        try:
            if dataset_info["type"] == "synthetic":
                # Generate synthetic golden set
                GOLDEN_TEXTS = [
                    "Invoice #12345",
                    "Total: €1,234.56",
                    "Patient: Max Mustermann",
                    "Date: 15.01.2026",
                    "APPROVED",
                    "Rechnung Nr. 98765",
                    "Betrag: 543,21 €",
                    "Dr. med. Schmidt",
                ]
                current_samples = [
                    {"image": create_text_image(text), "text": text}
                    for text in GOLDEN_TEXTS[:n_samples]
                ]
            else:
                # Load from HuggingFace - handle different dataset structures
                ds = load_dataset(dataset_info["id"], split="train", streaming=True)
                current_samples = []

                # Get dataset-specific keys (or use defaults)
                image_key = dataset_info.get("image_key", "image")
                text_key = dataset_info.get("text_key", "text")

                for i, sample in enumerate(ds):
                    if i >= n_samples:
                        break

                    # Extract image
                    img = None
                    if image_key in sample:
                        img = sample[image_key]
                        if not isinstance(img, Image.Image):
                            try:
                                img = Image.open(img).convert("RGB")
                            except:
                                continue
                    elif "file" in sample:
                        try:
                            img = Image.open(sample["file"]).convert("RGB")
                        except:
                            continue
                    else:
                        continue

                    # Extract text - handle different formats
                    text = "N/A"
                    if text_key in sample:
                        text_data = sample[text_key]
                        if isinstance(text_data, list):
                            # XFUND/FUNSD: list of words
                            text = " ".join(str(w) for w in text_data[:30])  # First 30 words
                        elif isinstance(text_data, dict):
                            # CORD: JSON structure with gt_parse
                            if "gt_parse" in text_data:
                                text = str(text_data["gt_parse"])[:200]
                            else:
                                text = str(text_data)[:200]
                        else:
                            text = str(text_data)
                    elif "text" in sample:
                        text = str(sample["text"])
                    elif "ground_truth" in sample:
                        text = str(sample["ground_truth"])

                    current_samples.append({"image": img, "text": text})

            print(f"\n✅ Loaded {len(current_samples)} samples")

            # Display preview
            if current_samples:
                n_preview = min(3, len(current_samples))
                fig, axes = plt.subplots(1, n_preview, figsize=(12, 3))
                if n_preview == 1:
                    axes = [axes]
                for ax, sample in zip(axes, current_samples[:n_preview]):
                    ax.imshow(sample["image"])
                    ax.set_title(
                        f"'{sample['text'][:30]}..'"
                        if len(sample["text"]) > 30
                        else f"'{sample['text']}'",
                        fontsize=9,
                    )
                    ax.axis("off")
                plt.tight_layout()
                plt.show()

        except Exception as e:
            print(f"\n❌ Error loading dataset: {e}")
            print("   Falling back to synthetic data...")
            current_samples = [
                {"image": create_text_image(text), "text": text}
                for text in ["Invoice #12345", "Total: €100.00", "APPROVED"]
            ]


load_data_btn.on_click(load_dataset_samples)

display(widgets.HBox([dataset_selector, sample_limit, load_data_btn]))
display(data_status)

print("Tip: Start with Synthetic, then progress to noisier datasets.")

## 3. Run OCR Inference

Now let's see what the model predicts on your loaded dataset.

In [ ]:
# =============================================================================
# OCR INFERENCE - Unified prediction for different model architectures
# =============================================================================


def predict_text(image: Image.Image, return_confidence: bool = True) -> tuple[str, float]:
    """Run OCR on a single image, return (text, confidence).

    Handles different model architectures:
    - TrOCR: Standard encoder-decoder
    - Florence-2: Prompt-based multi-task (Microsoft)
    - Qwen2-VL: Prompt-based multi-task (Alibaba)
    - Donut: Document understanding
    - Chandra: Heavy VLM
    """
    if model is None:
        raise ValueError("Load a model first! (Run the model selector cell)")

    # Ensure image is RGB (some datasets return grayscale)
    if image.mode != "RGB":
        image = image.convert("RGB")

    confidence = 0.0

    with torch.no_grad():
        if current_model_type == "trocr":
            # TrOCR: Standard OCR
            pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
            outputs = model.generate(
                pixel_values,
                max_length=64,
                num_beams=4,
                return_dict_in_generate=True,
                output_scores=return_confidence,
            )
            pred_text = processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]

            # Calculate confidence
            if return_confidence and outputs.scores:
                scores = torch.stack(outputs.scores, dim=1)
                probs = torch.softmax(scores, dim=-1)
                token_ids = outputs.sequences[0, 1:]
                if len(token_ids) > 0:
                    token_probs = probs[
                        0, range(min(len(token_ids), probs.shape[1])), token_ids[: probs.shape[1]]
                    ]
                    confidence = float(token_probs.mean())

        elif current_model_type == "florence":
            # Florence-2: Prompt-based
            model_info = MODELS[model_selector.value]
            prompt = model_info.get("task_prompt", "<OCR>")
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=256,
                num_beams=3,
            )
            generated_text = processor.batch_decode(outputs, skip_special_tokens=False)[0]
            # Parse Florence output
            parsed = processor.post_process_generation(
                generated_text, task=prompt, image_size=(image.width, image.height)
            )
            pred_text = parsed.get(prompt, generated_text)
            if isinstance(pred_text, dict):
                pred_text = str(pred_text)
            confidence = 0.8  # Florence doesn't provide easy confidence

        elif current_model_type == "qwen2vl":
            # Qwen2-VL: OCR mode - must explicitly ask for TEXT not detection
            prompt = "Please read and transcribe all the text visible in this image. Output only the text content, no coordinates or formatting."
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True).to(
                device
            )
            outputs = model.generate(**inputs, max_new_tokens=512)
            pred_text = processor.batch_decode(outputs, skip_special_tokens=True)[0]
            # Remove the prompt/assistant prefix from output
            if "assistant" in pred_text.lower():
                pred_text = pred_text.split("assistant")[-1].strip()
            if prompt in pred_text:
                pred_text = pred_text.replace(prompt, "").strip()
            confidence = 0.8

        elif current_model_type == "donut":
            # Donut: Document understanding
            pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
            decoder_input_ids = processor.tokenizer(
                "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
            ).input_ids.to(device)
            outputs = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=512,
                early_stopping=True,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
                num_beams=1,
                return_dict_in_generate=True,
            )
            pred_text = processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
            confidence = 0.8

        elif current_model_type == "chandra":
            # Chandra: Heavy VLM using chandra library
            from chandra.model.hf import generate_hf
            from chandra.model.schema import BatchInputItem
            from chandra.output import parse_markdown

            batch = [BatchInputItem(image=image, prompt_type="ocr_layout")]
            result = generate_hf(batch, model)[0]
            pred_text = parse_markdown(result.raw)
            confidence = 0.9  # Chandra is usually accurate

        elif current_model_type == "got_ocr":
            # GOT-OCR2: General OCR Theory - very capable document OCR
            # Uses chat-style inference
            pred_text = model.chat(processor, image, ocr_type="ocr")
            confidence = 0.85

        else:
            raise ValueError(f"Unknown model type: {current_model_type}")

    return pred_text, confidence


# =============================================================================
# RUN INFERENCE ON LOADED DATASET
# =============================================================================

if not current_samples:
    print("⚠️ Load a dataset first! (Run section 2)")
else:
    print(f"Running OCR on {len(current_samples)} samples...\n")
    print(f"{'Ground Truth':<35} | {'Prediction':<35} | CER   | Conf")
    print("-" * 95)

    results = []
    for sample in current_samples:
        gt_text = sample["text"]
        img = sample["image"]

        pred_text, conf = predict_text(img)
        cer = compute_cer(gt_text, pred_text)
        results.append({"gt": gt_text, "pred": pred_text, "cer": cer, "conf": conf, "image": img})

        # Truncate for display
        gt_display = gt_text[:32] + "..." if len(gt_text) > 35 else gt_text
        pred_display = pred_text[:32] + "..." if len(pred_text) > 35 else pred_text

        # Color code: green=perfect, yellow=close, red=bad
        status = "✅" if cer == 0 else "⚠️" if cer < 0.1 else "❌"
        print(f"{gt_display:<35} | {pred_display:<35} | {cer:.3f} | {conf:.2f} {status}")

    avg_cer = np.mean([r["cer"] for r in results])
    perfect_rate = sum(1 for r in results if r["cer"] == 0) / len(results)

    print("\nResults Summary:")
    print(f"  Average CER: {avg_cer:.3f}")
    print(f"  Perfect matches: {perfect_rate * 100:.1f}%")
    print(f"  Model: {model_selector.value}")
    print(f"  Dataset: {dataset_selector.value}")

## 4. Robustness Testing

**Key insight**: Real-world documents are messy. Test what breaks OCR:
- Blur (fax machines, bad scans)
- Rotation (misaligned scanning)
- Compression artifacts (JPEG quality)

This is how you build a **robust training set** - augment your golden set.

In [ ]:
# =============================================================================
# INTERACTIVE ROBUSTNESS TESTER
# =============================================================================

# Sliders for perturbation levels
blur_slider = widgets.FloatSlider(value=0, min=0, max=5, step=0.5, description="Blur σ:")
rotation_slider = widgets.FloatSlider(value=0, min=-15, max=15, step=1, description="Rotation °:")
jpeg_slider = widgets.IntSlider(value=95, min=10, max=95, step=5, description="JPEG Q:")

# Sample selector - dynamically populated from current_samples
robustness_sample_selector = widgets.Dropdown(
    options=[(s["text"][:40], i) for i, s in enumerate(current_samples)]
    if current_samples
    else [("Load data first", 0)],
    description="Sample:",
    layout=widgets.Layout(width="350px"),
)

test_output = widgets.Output()


def run_robustness_test(change=None):
    with test_output:
        clear_output(wait=True)

        if not current_samples:
            print("Load a dataset first (Section 2)")
            return

        idx = robustness_sample_selector.value
        sample = current_samples[idx]
        gt_text = sample["text"]
        original_img = sample["image"]

        # Apply perturbations
        img = original_img.copy()
        if blur_slider.value > 0:
            img = apply_gaussian_blur(img, sigma=blur_slider.value)
        if rotation_slider.value != 0:
            img = apply_rotation(img, degrees=rotation_slider.value)
        if jpeg_slider.value < 95:
            img = apply_jpeg_compression(img, quality=jpeg_slider.value)

        # Run prediction
        pred_text, conf = predict_text(img)
        cer = compute_cer(gt_text, pred_text)

        # Display
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].imshow(original_img)
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(img)
        axes[1].set_title(
            f"Perturbed (blur={blur_slider.value}, rot={rotation_slider.value}°, jpeg={jpeg_slider.value})"
        )
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

        print(f"\nGround Truth: {gt_text[:60]}{'...' if len(gt_text) > 60 else ''}")
        print(f"Prediction:   {pred_text[:60]}{'...' if len(pred_text) > 60 else ''}")
        print(f"CER: {cer:.3f} | Confidence: {conf:.2f}")

        if cer == 0:
            print("✅ Perfect match")
        elif cer < 0.1:
            print("⚠️ Close, but not perfect")
        else:
            print("❌ Significant errors - OCR is breaking")


# Connect sliders
for slider in [blur_slider, rotation_slider, jpeg_slider, robustness_sample_selector]:
    slider.observe(run_robustness_test, names="value")

run_btn = widgets.Button(description="Test Robustness", button_style="warning")
run_btn.on_click(run_robustness_test)

display(
    widgets.VBox(
        [
            widgets.HBox([robustness_sample_selector, run_btn]),
            widgets.HBox([blur_slider, rotation_slider, jpeg_slider]),
        ]
    )
)
display(test_output)

print("Adjust sliders and click 'Test Robustness' to see how perturbations affect OCR")

## 5. MLflow Experiment Tracking

### Setup Required
Before running this section, start MLflow server in a separate terminal:

```bash
cd /path/to/ml_portfolio
uv run mlflow server --host 0.0.0.0 --port 5000
```

### What Gets Tracked
| Item | Location |
|------|----------|
| **Parameters** | Model name, dataset, blur/rotation/jpeg settings |
| **Metrics** | CER mean, CER std, confidence, perfect match rate |
| **UI** | http://localhost:5000 |
| **Storage** | Local `mlruns/` folder (gitignored) |

### Data Flow
```
Notebook → MLflow Server (port 5000) → mlruns/ folder
                ↓
        UI at http://localhost:5000
```

In [ ]:
# =============================================================================
# MLFLOW INTEGRATION - Track your experiments
# =============================================================================
import mlflow

# Connect to local MLflow server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("ocr-model-comparison")


def run_tracked_experiment(
    experiment_name: str, blur: float = 0, rotation: float = 0, jpeg_quality: int = 95
):
    """Run a full experiment and log to MLflow."""

    if not current_samples:
        print("Load a dataset first")
        return

    with mlflow.start_run(run_name=experiment_name):
        # Log parameters
        mlflow.log_param("model", model_selector.value)
        mlflow.log_param("dataset", dataset_selector.value)
        mlflow.log_param("blur_sigma", blur)
        mlflow.log_param("rotation_deg", rotation)
        mlflow.log_param("jpeg_quality", jpeg_quality)
        mlflow.log_param("n_samples", len(current_samples))

        # Run evaluation
        all_cer = []
        all_conf = []

        for sample in current_samples:
            gt_text = sample["text"]
            original_img = sample["image"]

            # Apply perturbations
            img = original_img.copy()
            if blur > 0:
                img = apply_gaussian_blur(img, sigma=blur)
            if rotation != 0:
                img = apply_rotation(img, degrees=rotation)
            if jpeg_quality < 95:
                img = apply_jpeg_compression(img, quality=jpeg_quality)

            pred_text, conf = predict_text(img)
            cer = compute_cer(gt_text, pred_text)
            all_cer.append(cer)
            all_conf.append(conf)

        # Log metrics
        mlflow.log_metric("cer_mean", np.mean(all_cer))
        mlflow.log_metric("cer_std", np.std(all_cer))
        mlflow.log_metric("conf_mean", np.mean(all_conf))
        mlflow.log_metric("perfect_match_rate", sum(1 for c in all_cer if c == 0) / len(all_cer))

        print(f"✅ Logged experiment '{experiment_name}' to MLflow")
        print(f"   CER: {np.mean(all_cer):.3f} ± {np.std(all_cer):.3f}")
        print(f"   Perfect matches: {sum(1 for c in all_cer if c == 0)}/{len(all_cer)}")


# Example: Run a baseline experiment (only if data is loaded)
if current_samples:
    print("Running baseline experiment...")
    run_tracked_experiment("baseline-clean-images")
else:
    print("Load a dataset and model first, then run this cell to log experiments")

In [ ]:
# =============================================================================
# ROBUSTNESS SWEEP - Test multiple perturbation levels
# =============================================================================

# This runs multiple experiments to find where OCR breaks down
# Check MLflow UI at http://localhost:5000 to compare results!

print("Running robustness sweep...")
print("(This will log multiple experiments to MLflow)\n")

# Test blur levels
for blur in [0.5, 1.0, 2.0, 3.0]:
    run_tracked_experiment(f"blur-sigma-{blur}", blur=blur)

# Test rotation levels
for rotation in [5, 10, 15]:
    run_tracked_experiment(f"rotation-{rotation}deg", rotation=rotation)

# Test JPEG compression
for quality in [75, 50, 25]:
    run_tracked_experiment(f"jpeg-quality-{quality}", jpeg_quality=quality)

print("\nSweep complete. Check MLflow UI to compare experiments.")

## 6. Next Steps

Now that you can compare models and datasets, here's what to explore:

### Immediate Experiments
1. **Compare models** - Which is better: Florence-2 vs TrOCR vs Donut?
2. **Dataset progression** - How does CER change from Synthetic → SROIE → scanned_receipts?
3. **Find breaking points** - What blur level causes CER > 0.1 for each model?
4. **Check MLflow** - Compare experiments at http://localhost:5000

### Cascade Pattern (Production)
```
Fast Model (Florence-2) → Confidence < 0.8? → Heavy Model (Chandra)
```

### Next Notebook (02_fine_tuning.ipynb)
- Fine-tune on your specific document types
- Use Optuna for hyperparameter search
- Train with augmented data for robustness

### Full Pipeline Integration
- Document classification BEFORE OCR
- Text region segmentation (YOLO, SAM, Florence-2 with `<OD>`)
- Downstream understanding (MedGemma for medical, LayoutLMv3 for forms)

In [ ]:
# =============================================================================
# BONUS: Test your own images
# =============================================================================

# Drop an image path here to test on real data
# image_path = "path/to/your/image.png"
# img = Image.open(image_path).convert("RGB")
# pred, conf = predict_text(img)
# print(f"Prediction: {pred}")
# print(f"Confidence: {conf:.2f}")

print("Uncomment the code above and provide your own image path to test.")